In [7]:
# time series clustering
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from profiling import *

DATASET_PATH = "../data/datasets/train.parquet"

In [8]:
df = load_dataset(DATASET_PATH)

df

,seq_ix,step_in_seq,need_prediction,p0,p1,p2,p3,p4,p5,p6,...,dp0,dp1,dp2,dp3,dv0,dv1,dv2,dv3,t0,t1
0,0,0,0,1.077446,-0.710366,0.373591,0.835558,0.050204,-0.139710,1.367384,...,-0.031078,-0.053527,-0.475214,-0.410787,0.805692,0.516635,-0.368215,-0.503497,-0.564226,1.864018
1,0,1,0,1.100140,-0.562796,0.483658,0.875354,0.138444,-0.010037,1.354714,...,0.950145,-0.083598,-0.604252,-0.703991,-1.925171,1.252957,-1.563974,-1.929990,-0.239079,2.168844
2,0,2,0,1.100140,-0.562796,0.483658,0.875354,0.138444,-0.010037,1.870266,...,-0.168259,1.632066,-0.178565,0.121240,1.377339,1.803970,-0.402838,1.042579,2.237778,2.459723
3,0,3,0,0.955471,-1.486537,0.230353,0.676854,-0.262686,-0.366873,2.011996,...,-0.426942,-0.743530,0.680013,1.394857,0.785266,1.424660,1.258192,0.914074,1.998699,2.272444
4,0,4,0,1.025123,-0.832006,0.317650,0.759677,-0.062770,-0.230353,2.174650,...,0.195700,0.293986,0.494971,0.267884,-1.315502,1.775372,1.302424,1.355667,1.998699,2.324245
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10720995,10720,995,1,1.025123,-0.832006,0.317650,0.759677,-0.062770,-0.230353,1.218002,...,-0.094232,-0.193143,-0.436240,-0.165095,-0.985610,0.621099,-0.855287,-1.116369,0.086068,0.200428
10720996,10720,996,1,1.025123,-0.832006,0.317650,0.759677,-0.062770,-0.230353,1.218002,...,-0.094232,-0.193143,-0.436240,-0.165095,0.805692,0.516635,-0.368215,-0.503497,0.086068,0.210389
10720997,10720,997,1,1.025123,-0.832006,0.317650,0.759677,-0.062770,-0.230353,1.218002,...,-0.234219,-0.855287,0.520661,0.318969,0.482248,0.132113,0.048948,0.076604,0.086068,0.220351
10720998,10720,998,1,1.025123,-0.832006,0.317650,0.759677,-0.062770,-0.230353,1.218002,...,0.040157,-0.321611,0.408805,-0.273089,-0.181652,-0.181652,-0.181652,-0.181652,0.086068,0.230313


In [9]:
from feature_engineering import add_engineered_features, ENGINEERED_FEATURES

df = add_engineered_features(df)
print("Added engineered features:", ENGINEERED_FEATURES)

Added engineered features: ['spread', 'mid_price', 'bid_ask_volume_imbalance', 'depth_imbalance_level_0', 'depth_imbalance_level_1', 'depth_imbalance_level_2', 'depth_imbalance_level_3', 'depth_imbalance_level_4', 'depth_imbalance_level_5', 'price_momentum', 'volume_momentum', 'bid_ask_momentum_diff']


In [ ]:
sequences_array = np.array_split(df[["t0", "t1"] + [f"p{i}" for i in [0, 1, 2, 5, 6, 7, 8, 11]] + [f"dp{i}" for i in range(2)] + [f"dv{i}" for i in range(2)] + ENGINEERED_FEATURES + ['seq_ix']], df['seq_ix'].nunique())

sequences_array

[array([[ 1.07744551, -0.71036553,  0.37359089, ...,         nan,
                 nan,  0.        ],
        [ 1.10013962, -0.56279635,  0.48365796, ..., -2.70709634,
          6.44782782,  0.        ],
        [ 1.10013962, -0.56279635,  0.48365796, ...,  3.11342812,
         -3.30663371,  0.        ],
        ...,
        [ 1.46418595, -1.24477839,  0.46261042, ...,  1.87350202,
         -3.32263398,  0.        ],
        [ 1.91722083, -2.14146352,  0.34767547, ...,  0.01504111,
          2.00873566,  0.        ],
        [ 2.12239408, -1.75617325,  0.36298236, ...,  3.93769217,
         -2.89057493,  0.        ]], shape=(1000, 45)),
 array([[ 1.06930768, -0.54961628,  0.46098065, ...,         nan,
                 nan,  1.        ],
        [ 1.15175927, -0.89942819,  0.52052516, ...,  2.40365314,
         -7.18417549,  1.        ],
        [ 1.13051748, -1.26966441,  0.40880513, ..., -5.93439674,
          7.53697586,  1.        ],
        ...,
        [ 1.06930768, -0.54961628,  

In [11]:
len(sequences_array), sequences_array[0].shape

(10721, (1000, 45))